# MOMENT anomaly scoring — DIMER live tutorial

This notebook demonstrates **raw reconstruction-residual ranking**, not a binary detector. The sample contains three documented synthetic spikes in the `vibration` channel. MOMENT scores each visible, non-padded position with an unmasked self-reconstruction residual.

**There is no universal threshold in v1.** High scores may be useful for ranking or downstream calibration on a domain-specific reference segment, but this tutorial does not convert them into anomaly labels.


## 1. Bootstrap the repository and locked runtime


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kurtvalcorza/moment-pipeline.git"
REPO_NAME = "moment-pipeline"
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    if not Path(REPO_NAME).exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL], check=True)
    os.chdir(REPO_NAME)
    ROOT = Path.cwd()
    subprocess.run(["uv", "pip", "install", "--system", "-r", "requirements.lock.txt"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "--no-deps", "-e", "."], check=True)
else:
    print(f"Repository checkout detected: {ROOT}")


## 2. Generate and verify the labelled synthetic demonstration sample


In [ ]:
import hashlib
import json

import pandas as pd

sample_root = ROOT / "examples" / "sample-data"
subprocess.run([sys.executable, str(sample_root / "generate_samples.py")], check=True)
sample_path = sample_root / "moment_anomaly.csv"
label_path = sample_root / "moment_anomaly_labels.csv"
manifest = {}
for line in (sample_root / "SHA256SUMS").read_text(encoding="utf-8").splitlines():
    digest, filename = line.split("  ", 1)
    manifest[filename] = digest
for path in (sample_path, label_path):
    observed = hashlib.sha256(path.read_bytes()).hexdigest()
    assert observed == manifest[path.name]
frame = pd.read_csv(sample_path)
labels = pd.read_csv(label_path)
frame["timestamp"] = pd.to_datetime(frame["timestamp"])
labels["timestamp"] = pd.to_datetime(labels["timestamp"])
print(f"verified sample rows={len(frame)}, injected anomalies={int(labels['is_injected_anomaly'].sum())}")


## 3. Canonicalize and compute raw anomaly scores


In [ ]:
from moment_pipeline import build_provenance, load_moment, score_anomalies, to_windows

windows = to_windows(frame)
model = load_moment(task="reconstruction", device="cpu")
result = score_anomalies(windows, model, loss="mae", channel_aggregation="none", warmup=False)
provenance = build_provenance(model, windows, result)
scores = result.to_frame()
print("score policy:", result.score_policy)
print("threshold policy:", result.threshold_policy)
print("scored fraction:", result.scored_point_fraction)


## 4. Rank the injected demonstration points


In [ ]:
vibration = scores[(scores["channel"] == "vibration") & scores["scored"]].copy()
ranked = vibration.merge(labels, on=["series_id", "timestamp"], how="left")
ranked["is_injected_anomaly"] = ranked["is_injected_anomaly"].fillna(False).astype(bool)
ranked = ranked.sort_values("anomaly_score", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1
print(ranked[["rank", "timestamp", "anomaly_score", "is_injected_anomaly"]].head(12))
print("injected ranks:", ranked.loc[ranked["is_injected_anomaly"], ["timestamp", "rank", "anomaly_score"]].to_dict("records"))


## 5. Visualize raw score over time


In [ ]:
def write_line_svg(path, layers, *, title, width=760, height=280):
    all_values = [float(value) for _, values in layers for value in values]
    low, high = min(all_values), max(all_values)
    span = high - low or 1.0
    max_points = max(len(values) for _, values in layers)
    left, right, top, bottom = 48, width - 20, 30, height - 38
    def point(index, value):
        x = left + (right - left) * index / max(max_points - 1, 1)
        y = bottom - (bottom - top) * (float(value) - low) / span
        return f"{x:.1f},{y:.1f}"
    svg = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">', f'<text x="{left}" y="18" font-family="sans-serif" font-size="14">{title}</text>']
    for idx, (label, values) in enumerate(layers):
        points = " ".join(point(i, value) for i, value in enumerate(values))
        stroke = ["#111827", "#2563eb", "#dc2626"][idx % 3]
        svg.append(f'<polyline fill="none" stroke="{stroke}" stroke-width="2" points="{points}"/>')
        svg.append(f'<text x="{left + 180 * idx}" y="{height - 10}" font-family="sans-serif" font-size="12" fill="{stroke}">{label}</text>')
    svg.append("</svg>")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(svg), encoding="utf-8")
    return path

ordered = vibration.sort_values("timestamp")
score_plot = write_line_svg(ROOT / "outputs" / "moment_anomaly_scores.svg", [("raw MAE residual", ordered["anomaly_score"].fillna(0.0).tolist())], title="MOMENT raw anomaly score — vibration")
try:
    from IPython.display import SVG, display
    display(SVG(filename=str(score_plot)))
except ImportError:
    print(f"SVG written to {score_plot}")


## 6. Export raw scores and provenance


In [ ]:
output_dir = ROOT / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
scores.to_csv(output_dir / "moment_anomaly_scores.csv", index=False)
(output_dir / "moment_anomaly_provenance.json").write_text(json.dumps(provenance, indent=2, default=str), encoding="utf-8")
print("exports:", sorted(path.name for path in output_dir.glob("moment_anomaly*")))


## Interpretation

The injected spikes are included only to make the ranking demonstration falsifiable. They are **not a calibration dataset** and this notebook does not derive a threshold from them. The shipped output is the raw residual plus explicit scored-domain/provenance metadata.
